In [1]:
# Navigate to the Drive folder synced with your Repo
# Assuming it's mounted. If not, mount it first.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# CELL 1: SETUP AND GITHUB PULL
import os
from google.colab import userdata

# Install required geospatial and metrics libraries
!pip install -q rasterio torchmetrics

# --- GITHUB CONFIGURATION ---
GITHUB_USER = "MagicMorgigi"  # <-- Change this
REPO_NAME = "master_thesis_FM"          # <-- Change this
BRANCH = "main"                     # <-- Change this

# Retrieve PAT from Colab Secrets
PAT = userdata.get('MA_colab_github_PAT')
REPO_URL = f"https://{PAT}@github.com/{GITHUB_USER}/{REPO_NAME}.git"


REPO_PATH = f"/content/drive/MyDrive/Colab Notebooks/{REPO_NAME}" # <-- Adjust to your GDrive sync path
os.chdir(REPO_PATH)

print(f"Current Directory: {os.getcwd()}")

# Configure git, fetch, checkout branch, and pull updates (no cloning)
!git config --global user.email "colab-worker@example.com"
!git config --global user.name "Colab Pro Pipeline"
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}
print(f"Successfully pulled latest changes from branch: {BRANCH}")

In [3]:
!pip install -q torchmetrics rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 40.6 MB/s eta 0:00:00


In [4]:
# CELL 2: IMPORTS AND SEEDING
import os
import gc
import re
import csv
import torch
import shutil
import rasterio
import subprocess
import numpy as np
import pandas as pd
import torch.nn as nn
import matplotlib.pyplot as plt
from glob import glob
from tqdm.auto import tqdm
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from torchmetrics import MeanSquaredError, MeanAbsoluteError, R2Score

def seed_everything(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"Global seed set to: {seed}")

SEED = 42
seed_everything(SEED)

Global seed set to: 42


In [5]:
# CELL 3: CONFIGURATION
@dataclass(frozen=True)
class Config:
    # Google Drive Archive Paths (Adjust these to your Drive locations)
    DRIVE_ROOT: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives"

    DRIVE_FEAT_TRAIN: str = f"{DRIVE_ROOT}/AnySat_embedded_dense_features/6%_sampled/train.tar.gz"
    DRIVE_FEAT_VAL: str = f"{DRIVE_ROOT}/AnySat_embedded_dense_features/6%_sampled/val.tar.gz"
    DRIVE_FEAT_TEST: str = f"{DRIVE_ROOT}/AnySat_embedded_dense_features/6%_sampled/test.tar.gz"
    DRIVE_LAB_TRAIN: str = f"{DRIVE_ROOT}/labels/6%_sampled/train.tar.gz"
    DRIVE_LAB_VAL: str = f"{DRIVE_ROOT}/labels/6%_sampled/val.tar.gz"
    DRIVE_LAB_TEST: str = f"{DRIVE_ROOT}/labels/6%_sampled/test.tar.gz"

    # Local SSD Extraction Paths
    LOCAL_DATA_DIR: str = "/content/local_data"

    # Training Parameters
    IN_CHANNELS: int = 1536
    BATCH_SIZE: int = 1      # Kept low due to 196MB file sizes
    NUM_WORKERS: int = 0
    LEARNING_RATE: float = 1e-3
    WEIGHT_DECAY: float = 1e-5
    MAX_EPOCHS: int = 100
    PATIENCE: int = 10       # Early stopping
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    PATCH_SIZE = 40

    # Drive Output Path
    EXP_NAME = f"AnySat_LP_DENSE_patchSize{PATCH_SIZE}_6%subset_BS{BATCH_SIZE}_LR{str(LEARNING_RATE).split('.')[1]}_EP{MAX_EPOCHS}_SEED{SEED}_L4"
    OUTPUT_DIR: str = f"/content/drive/MyDrive/master_thesis_FM/experiments/AnySat/linear_probing/dense_features/{EXP_NAME}"

CONFIG = Config()
os.makedirs(CONFIG.OUTPUT_DIR, exist_ok=True)
print(f"Configuration locked. Target Device: {CONFIG.DEVICE}")

Configuration locked. Target Device: cuda


In [6]:
# CELL 4: DATA TRANSFER AND EXTRACTION
def extract_archive_sequentially(drive_path: str, target_dir: str):
    """Copies, extracts, and deletes an archive to minimize SSD footprint."""
    if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
        print(f"Data already present in {target_dir}. Skipping extraction.")
        return

    os.makedirs(target_dir, exist_ok=True)
    archive_name = os.path.basename(drive_path)
    local_archive_path = os.path.join(CONFIG.LOCAL_DATA_DIR, archive_name)

    print(f"Copying {archive_name} from Drive...")
    shutil.copy(drive_path, local_archive_path)

    print(f"Extracting {archive_name} into {target_dir}...")
    # subprocess with tar is significantly faster than Python's tarfile
    subprocess.run(["tar", "-xzf", local_archive_path, "-C", target_dir], check=True)

    print(f"Deleting local archive {local_archive_path} to free space...")
    os.remove(local_archive_path)
    print("Done.\n")

# Define target directories on local SSD
local_paths = {
    'feat_train': os.path.join(CONFIG.LOCAL_DATA_DIR, 'feat_train'),
    'feat_val': os.path.join(CONFIG.LOCAL_DATA_DIR, 'feat_val'),
    'feat_test': os.path.join(CONFIG.LOCAL_DATA_DIR, 'feat_test'),
    'lab_train': os.path.join(CONFIG.LOCAL_DATA_DIR, 'lab_train'),
    'lab_val': os.path.join(CONFIG.LOCAL_DATA_DIR, 'lab_val'),
    'lab_test': os.path.join(CONFIG.LOCAL_DATA_DIR, 'lab_test')
}

extract_archive_sequentially(CONFIG.DRIVE_FEAT_TRAIN, local_paths['feat_train'])
extract_archive_sequentially(CONFIG.DRIVE_FEAT_VAL, local_paths['feat_val'])
extract_archive_sequentially(CONFIG.DRIVE_FEAT_TEST, local_paths['feat_test'])
extract_archive_sequentially(CONFIG.DRIVE_LAB_TRAIN, local_paths['lab_train'])
extract_archive_sequentially(CONFIG.DRIVE_LAB_VAL, local_paths['lab_val'])
extract_archive_sequentially(CONFIG.DRIVE_LAB_TEST, local_paths['lab_test'])

Copying train.tar.gz from Drive...
Extracting train.tar.gz into /content/local_data/feat_train...
Deleting local archive /content/local_data/train.tar.gz to free space...
Done.

Copying val.tar.gz from Drive...
Extracting val.tar.gz into /content/local_data/feat_val...
Deleting local archive /content/local_data/val.tar.gz to free space...
Done.

Copying test.tar.gz from Drive...
Extracting test.tar.gz into /content/local_data/feat_test...
Deleting local archive /content/local_data/test.tar.gz to free space...
Done.

Copying train.tar.gz from Drive...
Extracting train.tar.gz into /content/local_data/lab_train...
Deleting local archive /content/local_data/train.tar.gz to free space...
Done.

Copying val.tar.gz from Drive...
Extracting val.tar.gz into /content/local_data/lab_val...
Deleting local archive /content/local_data/val.tar.gz to free space...
Done.

Copying test.tar.gz from Drive...
Extracting test.tar.gz into /content/local_data/lab_test...
Deleting local archive /content/local_

In [21]:
# CELL 5: DATASET AND DATALOADERS (STRICT GATEKEEPER)
class BiomassDataset(Dataset):
    def __init__(self, feat_dir: str, lab_dir: str):
        self.feat_paths = glob(os.path.join(feat_dir, "**/*.pt"), recursive=True)
        self.samples = []

        # Regex to extract 8-character ID
        feat_pattern = re.compile(r'_([A-Za-z0-9]{8})_S1_10\.pt$')

        for f_path in self.feat_paths:
            # Drop obvious tiny metadata files
            if os.path.getsize(f_path) < 100_000_000:
                continue

            match = feat_pattern.search(os.path.basename(f_path))
            if match:
                file_id = match.group(1)
                expected_lab = os.path.join(lab_dir, f"{file_id}_agbm.tif")
                if os.path.exists(expected_lab):
                    self.samples.append((f_path, expected_lab, file_id))

        print(f"Mapped {len(self.samples)} valid dense feature-label pairs from {feat_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        f_path, l_path, file_id = self.samples[idx]

        # 1. Load feature
        feat = torch.load(f_path, map_location='cpu')

        if feat.is_sparse:
            feat = feat.to_dense()

        feat = feat.to(torch.float32)

        # 2. Strict Shape Gatekeeper
        if feat.dim() == 4 and feat.shape == (1, 256, 256, 1536):
            feat = feat.squeeze(0).permute(2, 0, 1)
        elif feat.dim() == 4 and feat.shape == (1, 1536, 256, 256):
            feat = feat.squeeze(0)
        elif feat.dim() == 3 and feat.shape == (256, 256, 1536):
            feat = feat.permute(2, 0, 1)
        elif feat.dim() == 3 and feat.shape == (1536, 256, 256):
            pass # Already perfectly shaped
        elif feat.dim() == 2 or feat.dim() == 1:
            if feat.numel() == 100663296:
                feat = feat.reshape(1536, 256, 256)
            else:
                raise ValueError(f"\n[DATA ERROR] File contains a flattened tensor of {feat.numel()} elements.\nFile: {f_path}")
        else:
            raise ValueError(f"\n[DATA ERROR] File contains a {feat.dim()}D tensor of shape {feat.shape}.\nFile: {f_path}\nFile Size: {os.path.getsize(f_path) / (1024**2):.2f} MB")

        # Ultimate Pipeline Safety Net
        if feat.shape != (1536, 256, 256):
            raise ValueError(f"\n[PIPELINE ERROR] Formatting failed. Yielded shape {feat.shape}.\nFile: {f_path}")

        # 3. Load label via rasterio
        with rasterio.open(l_path) as src:
            lab = src.read(1)
            lab = torch.from_numpy(lab).to(torch.float32).unsqueeze(0)

        return feat, lab, file_id

train_ds = BiomassDataset(local_paths['feat_train'], local_paths['lab_train'])
val_ds = BiomassDataset(local_paths['feat_val'], local_paths['lab_val'])
test_ds = BiomassDataset(local_paths['feat_test'], local_paths['lab_test'])

train_loader = DataLoader(train_ds, batch_size=CONFIG.BATCH_SIZE, shuffle=True, num_workers=CONFIG.NUM_WORKERS, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=CONFIG.BATCH_SIZE, shuffle=False, num_workers=CONFIG.NUM_WORKERS, pin_memory=False)
test_loader = DataLoader(test_ds, batch_size=CONFIG.BATCH_SIZE, shuffle=False, num_workers=CONFIG.NUM_WORKERS, pin_memory=False)

Mapped 417 valid dense feature-label pairs from /content/local_data/feat_train
Mapped 105 valid dense feature-label pairs from /content/local_data/feat_val
Mapped 167 valid dense feature-label pairs from /content/local_data/feat_test


In [18]:
# CELL 6: MODEL ARCHITECTURE AND METRICS
class LinearProbe(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        # 1x1 Conv acts as a pixel-wise linear layer
        self.probe = nn.Conv2d(in_channels, 1, kernel_size=1, bias=True)

    def forward(self, x):
        return self.probe(x)

model = LinearProbe(in_channels=CONFIG.IN_CHANNELS).to(CONFIG.DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG.LEARNING_RATE, weight_decay=CONFIG.WEIGHT_DECAY)

# Global Metric Accumulators (Run on Device to save RAM)
train_mse = MeanSquaredError().to(CONFIG.DEVICE)
val_mse = MeanSquaredError().to(CONFIG.DEVICE)
val_rmse = MeanSquaredError(squared=False).to(CONFIG.DEVICE)
val_mae = MeanAbsoluteError().to(CONFIG.DEVICE)
val_r2 = R2Score().to(CONFIG.DEVICE)

In [22]:
# CELL 7: TRAINING LOOP WITH CHECKPOINTING & TRIPWIRE
checkpoint_path = os.path.join(CONFIG.OUTPUT_DIR, "checkpoint.pth")
history_path = os.path.join(CONFIG.OUTPUT_DIR, "training_history.csv")
best_model_path = os.path.join(CONFIG.OUTPUT_DIR, "best_linear_probe.pth")

start_epoch = 0
best_val_loss = float('inf')
patience_counter = 0
history = []

if os.path.exists(checkpoint_path):
    print("Found existing checkpoint. Resuming training...")
    ckpt = torch.load(checkpoint_path, map_location=CONFIG.DEVICE)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['best_val_loss']
    patience_counter = ckpt['patience_counter']
    if os.path.exists(history_path):
        history = pd.read_csv(history_path).to_dict('records')

if not os.path.exists(history_path):
    with open(history_path, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Epoch', 'Train_MSE', 'Val_MSE', 'Val_RMSE', 'Val_MAE', 'Val_R2'])

for epoch in range(start_epoch, CONFIG.MAX_EPOCHS):
    model.train()
    train_mse.reset()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG.MAX_EPOCHS} [Train]")
    for feats, labs, file_ids in pbar:

        # --- THE TRIPWIRE ---
        # A valid batched tensor should be [Batch, 1536, 256, 256].
        # A valid unbatched tensor should be [1536, 256, 256].
        if feats.dim() < 3:
            print(f"\n[TRIPWIRE TRIGGERED] The DataLoader yielded a corrupted tensor right before the model!")
            print(f"-> Responsible File ID: {file_ids}")
            print(f"-> Tensor Shape: {feats.shape}")
            print(f"-> Tensor Data: {feats}")
            raise RuntimeError("Pipeline stopped by Tripwire. See prints above.")
        # --------------------

        feats, labs = feats.to(CONFIG.DEVICE), labs.to(CONFIG.DEVICE)

        optimizer.zero_grad()
        preds = model(feats)

        loss = torch.nn.functional.mse_loss(preds.view(-1), labs.view(-1))
        loss.backward()
        optimizer.step()

        train_mse.update(preds.view(-1), labs.view(-1))
        pbar.set_postfix({'MSE': f"{loss.item():.4f}"})

    epoch_train_mse = train_mse.compute().item()

    # Validation
    model.eval()
    val_mse.reset()
    val_rmse.reset()
    val_mae.reset()
    val_r2.reset()

    with torch.no_grad():
        for feats, labs, _ in tqdm(val_loader, desc=f"Epoch {epoch+1}/{CONFIG.MAX_EPOCHS} [Val]", leave=False):
            feats, labs = feats.to(CONFIG.DEVICE), labs.to(CONFIG.DEVICE)
            preds = model(feats)

            p_flat, l_flat = preds.view(-1), labs.view(-1)
            val_mse.update(p_flat, l_flat)
            val_rmse.update(p_flat, l_flat)
            val_mae.update(p_flat, l_flat)
            val_r2.update(p_flat, l_flat)

    epoch_val_mse = val_mse.compute().item()
    epoch_val_rmse = val_rmse.compute().item()
    epoch_val_mae = val_mae.compute().item()
    epoch_val_r2 = val_r2.compute().item()

    print(f"Epoch {epoch+1} Metrics -> Train MSE: {epoch_train_mse:.4f} | Val MSE: {epoch_val_mse:.4f} | Val R2: {epoch_val_r2:.4f}")

    row = {'Epoch': epoch+1, 'Train_MSE': epoch_train_mse, 'Val_MSE': epoch_val_mse, 'Val_RMSE': epoch_val_rmse, 'Val_MAE': epoch_val_mae, 'Val_R2': epoch_val_r2}
    history.append(row)
    with open(history_path, mode='a', newline='') as f:
        csv.DictWriter(f, fieldnames=row.keys()).writerow(row)

    ckpt_state = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_val_loss': best_val_loss,
        'patience_counter': patience_counter
    }

    if epoch_val_mse < best_val_loss:
        best_val_loss = epoch_val_mse
        patience_counter = 0
        ckpt_state['best_val_loss'] = best_val_loss
        ckpt_state['patience_counter'] = 0
        torch.save(model.state_dict(), best_model_path)
        print("-> Best model saved!")
    else:
        patience_counter += 1

    torch.save(ckpt_state, checkpoint_path)

    gc.collect()
    torch.cuda.empty_cache()

    if patience_counter >= CONFIG.PATIENCE:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break

Epoch 1/100 [Train]:   0%|          | 0/417 [00:00<?, ?it/s]

ValueError: 
[DATA ERROR] File contains a 0D tensor of shape torch.Size([]).
File: /content/local_data/feat_train/Embedded_Features_OutputTypeDENSE_PatchSize40_asc_3x256x256_ff4ebe0a_S1_10.pt
File Size: 192.00 MB

In [ ]:
# CELL 8: INFERENCE AND VISUALIZATION
# This can be run independently after a session restart
# (provided Cell 2, 3, 5, and 6 are executed to define objects).

best_model_path = os.path.join(CONFIG.OUTPUT_DIR, "best_linear_probe.pth")
if not os.path.exists(best_model_path):
    raise FileNotFoundError("Best model not found. Train the model first.")

model.load_state_dict(torch.load(best_model_path, map_location=CONFIG.DEVICE))
model.eval()

test_mse = MeanSquaredError().to(CONFIG.DEVICE)
test_rmse = MeanSquaredError(squared=False).to(CONFIG.DEVICE)
test_mae = MeanAbsoluteError().to(CONFIG.DEVICE)
test_r2 = R2Score().to(CONFIG.DEVICE)

viz_samples = []

print("Running inference on Test Set...")
with torch.no_grad():
    for i, (feats, labs, ids) in enumerate(tqdm(test_loader, desc="Testing")):
        feats, labs = feats.to(CONFIG.DEVICE), labs.to(CONFIG.DEVICE)
        preds = model(feats)

        p_flat, l_flat = preds.view(-1), labs.view(-1)
        test_mse.update(p_flat, l_flat)
        test_rmse.update(p_flat, l_flat)
        test_mae.update(p_flat, l_flat)
        test_r2.update(p_flat, l_flat)

        # Store first batch for visualization
        if i == 0:
            for b in range(min(4, feats.size(0))): # Visualizing up to 4 images
                viz_samples.append({
                    'id': ids[b],
                    'target': labs[b].cpu().squeeze().numpy(),
                    'pred': preds[b].cpu().squeeze().numpy()
                })

final_metrics = {
    'Test_MSE': test_mse.compute().item(),
    'Test_RMSE': test_rmse.compute().item(),
    'Test_MAE': test_mae.compute().item(),
    'Test_R2': test_r2.compute().item()
}

# Save final metrics to text file
metrics_out_path = os.path.join(CONFIG.OUTPUT_DIR, 'test_metrics.txt')
with open(metrics_out_path, 'w') as f:
    f.write("--- BEST MODEL METRICS ---\n")
    for k, v in final_metrics.items():
        f.write(f"{k}: {v:.4f}\n")
    print(f"Metrics saved to {metrics_out_path}")
    print(final_metrics)

# Visualization
fig, axes = plt.subplots(len(viz_samples), 2, figsize=(10, 4 * len(viz_samples)))
for idx, sample in enumerate(viz_samples):
    # Determine common vmin and vmax for accurate color comparison
    vmin = min(sample['target'].min(), sample['pred'].min())
    vmax = max(sample['target'].max(), sample['pred'].max())

    # Target
    im1 = axes[idx, 0].imshow(sample['target'], cmap='viridis', vmin=vmin, vmax=vmax)
    axes[idx, 0].set_title(f"Target AGB - ID: {sample['id']}")
    axes[idx, 0].axis('off')

    # Prediction
    im2 = axes[idx, 1].imshow(sample['pred'], cmap='viridis', vmin=vmin, vmax=vmax)
    axes[idx, 1].set_title(f"Predicted AGB - ID: {sample['id']}")
    axes[idx, 1].axis('off')

fig.colorbar(im1, ax=axes.ravel().tolist(), orientation='horizontal', fraction=0.02, pad=0.04, label='AGB (tons)')
plt.savefig(os.path.join(CONFIG.OUTPUT_DIR, 'prediction_visualizations.png'))
plt.show()

In [ ]:
# CELL 9: GITHUB COMMIT AND PUSH
# Committing the Notebook assuming you've saved it via File > Save in Colab
!git add .
!git commit -m "Automated commit: completed pipeline run and updated outputs"
!git push origin {BRANCH}
print("Successfully pushed changes to GitHub.")

In [ ]:
# CELL 10: FLUSH DATA AND AUTOMATIC RUNTIME TERMINATION
import time
from google.colab import drive
from google.colab import runtime

print("Pipeline completed successfully.")
print("Flushing I/O buffers to Google Drive to prevent asynchronous data loss...")

# Force Google Drive to sync all pending writes and cleanly unmount
drive.flush_and_unmount()
print("Google Drive successfully synced and unmounted.")

print("Disconnecting and deleting the runtime in 10 seconds to preserve Compute Units...")
time.sleep(10)

# Unassign the runtime. Note: This will kick you out of the session.
runtime.unassign()